In [44]:

from dotenv import load_dotenv
from openai import OpenAI
import os
from pypdf import PdfReader
from pydantic import BaseModel

In [45]:

load_dotenv(override=True)

myOpenAI = OpenAI()

In [46]:

# 1 rzecz to pobór dokumentów pdf oraz txt

pdfFile = ''
pdfTxt = PdfReader('pzpPDF.pdf')
# pętla, plus metoda.extract_text
for page in pdfTxt.pages:
    text = page.extract_text()
    if (text):
        pdfFile += text


print(pdfFile)


# 2 to sposób na pobranie pliku txt w trybie read i encodint i przekazanie metodą read do zmienenj

with open('pzp.txt','r',encoding= 'utf-8') as f:
    txtFile = f.read()



print(txtFile)


Strona 1
Podstawy zarz■dzania projektami
dr Tomasz Kopczy■ski
Strona 2
Wyzwania
PM
Zarz■dzanie
sytuacjami
kryzysowymi
Zarz■dzanie
zespo■ami
rozproszonymi i
zró■nicowanymi
Zwinne i
elastyczne
zespo■y
Zarz■dzanie
kompetencjami
i emocjami
Strona 3
 PROJEKT– tymczasowe przedsi■wzi■cie, maj■ce na celu
stworzenie szczególnej warto■ci (produkt, efekt), gdzie
tymczasowo■■ oznacza, ■e przedsi■wzi■cie ma okre■lony
pocz■tek i koniec.
 ZARZ■DZANIE PROJEKTEM – dzia■anie, w trakcie którego
osoba kieruj■ca projektem (kierownik projektu) przeprowadza
wraz z zespo■em planowanie oraz realizuje i monitoruje projekt
stosuj■c odpowiednie metody i dokumenty, aby osi■gn■■
wyznaczony cel w okre■lonym czasie, zakresie, kosztach i
osi■gaj■c odpowiedni■ jako■■ ko■cow■ przedsi■wzi■cia.
Strona 4
4 edycja (2009)
460 stron
Kluczowe procesy,
produkty, narz■dzi
i techniki
6 edycja (2017)
760 stron
Elementy
podej■cia
zwinnego
5 edycja (2013)
590 stron
Nowy obszar –
zarz■dzanie
interesariuszami
7 edycja (2021)
290 str

In [47]:

# tworzymy system prompt dla każdego z modelu od 1 kóry generuje 1 odpowiedz  oraz ewentualnie dla 3 modelu rerun
# który uruchomi się jesli evaluioatr uzna 1 odpowiedz za błedą, wówczas powiększamy system_prompt dostępny feedback z evaluation 

system_prompt = f"""
Jesteś asystentem eksperckim wspierającym użytkownika w zrozumieniu zarządzania projektami.

Odpowiadasz na pytania na podstawie dostarczonych materiałów, szczególnie dotyczących:
- definicji: projektu i zarządzania projektem
- etapów projektu (inicjowanie, planowanie, realizacja, zamknięcie)
- trójkąta projektowego (zakres, czas, budżet, jakość)
- ryzyka, interesariuszy i ról w projekcie

Odpowiadaj jasno, konkretnie i w sposób uporządkowany.
Jeśli to możliwe, podawaj przykłady praktyczne.

Jeśli nie znasz odpowiedzi — powiedz to wprost.

---

## Materiały:

### Dokument:
{txtFile}

### PDF:
{pdfFile}

---

Na podstawie tego kontekstu odpowiadaj na pytania użytkownika.
"""

In [48]:
# teraz za pomocą biblioteki pydantic do ułozenia danych, a konkretnie modułu BaseModel, tworzymy klasę która 
# przyjmie odpowiedź modelu Evaluator - (kóry sluzy do weryfikacji odpowiedzi modelu 1). w formie obiektu, i zwróci nam odpowiedź w ułozony sposób
# po 1 da informacje true/false czy akceptuje odpowedz zgodnie z promptem czy tez nie, i po 2 da feedback dlaczego odrzucił odpowiedź 
# działa na podstawie user promptu, evaliaotr promptu i odpowiedzi 1 modelu ai

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [49]:
# tworzymy prompt dla evaluatora

# prompt dla evaluatora

evaluator_system_prompt = f"""
Jesteś ewaluatorem odpowiedzi AI.

Twoim zadaniem jest ocenić, czy odpowiedź jest zgodna z dostarczonymi materiałami dotyczącymi zarządzania projektami.

Weź pod uwagę:
- poprawność merytoryczną
- zgodność z koncepcjami (etapy projektu, trójkąt projektowy, ryzyko, interesariusze, role)
- czy odpowiedź jest jasna i użyteczna

Jeśli odpowiedź zawiera błędy, nieścisłości lub wychodzi poza kontekst — oznacz ją jako nieakceptowalną.

---

## Materiały:

### Dokument:
{txtFile}
### PDF:
{pdfFile}

---

Zwróć WYŁĄCZNIE JSON w formacie:
 is_acceptable (bool), feedback (string)
"""

In [50]:

# tworzymy funkcje która przyjmuje history, message i reply czyli historie romozwy w tablicy, message od usera i reply czyli odpowiedź 1 modelu chat
# funkcja zostaje uruchomiona w modelu evalautor - modelu sprawdzającym funkcja trafia do listy message jako wiadomosci usera, dlatego całosc zwracamy returnem

def evaluator_user_prompt(reply,message,history):
    prompt = f'tutaj masz całą historie rozmowy {history}'
    prompt += f'tutaj pytanie usera {message}'
    prompt += f' tutaj odpowiedź agenta {reply}'
    prompt += f'przeanalizuj odpowiedź i oceń ją'
    return prompt

In [51]:

# teraz piszemy właściwą funkcje evalutor, funkcja przyjmuje reply,message,history oraz tworzy model i zwraca odpowiedż do klasy evaluation
# która jest obiektem i po analizie odpowiedzi z moedlu evaluator odda nam dane czy akceputje odpwiedz czy nie i da feedbkack czmeu nie
# akcept idzie w boolean true/false 

def evaluator(reply,message,history) -> Evaluation:

    # wpierw tworzymy liste messages do modelu i tu wrzycamy evaluator system prompt oraz funkcje evaluator user prompt która ma distep do replu message,history
    messages = [{'role':'system','content':evaluator_system_prompt}] + [{'role':'user','content': evaluator_user_prompt(reply,message,history)}]

    # teraz tworzymy model ale taki króry zwróci odpowiedz jako obiekt do klasy Evaluation, zeby BaseModel z pydantica mogł przerobic nasza odpowiedz na potrzebne dane
    # true/false i dać feedback

    response = myOpenAI.beta.chat.completions.parse(

            model='gpt-4.1-mini',
            messages=messages,
            # dodajemy response format który ułozy nam dzieki odpowiedzi konktertny format tryu/fals i feedback
            response_format=Evaluation
    )

    return response.choices[0].message.parsed

In [52]:
# piszemy funkcje rerun która uruchamia się jedynie jesli Evalaution da odpowiedz false i wtedy w funkcji chat kóra ma tez 1 model odpowiedzi
# uruchomimy rerun

def rerun(reply,message,history,feedback):
    # budujemy nowy sytem prompt oparty o system prompt
    update_system_prompt = system_prompt + f'poprzednia wiadomosc zostala odrzucona'
    update_system_prompt += f'tutaj odpowiedź {reply}'
    update_system_prompt += f' a tutaj feedback {feedback}'

    # budujemy liste messages
    messages = [{'role':'system','content':update_system_prompt}] + history + [{'role':'user','content':message}]

    # budujemy response

    response = myOpenAI.chat.completions.create(
        
        model = 'gpt-4.1-mini',
        messages=messages
    )

    return response.choices[0].message.content

In [53]:
# teraz piszemy główną funkcje chat , funkcja przyj uje 2 wartosci message od usera i history

def chat(message,history):

    # wpierw piszemy warunek który będzie weryfikował na podstawie stringa usera, czy user_prompt ma zostać zaktauilozwamy o dodatkowe indstrukcje 
    # u nas jeśli w message znajdzie się słowo dokumentów wówczas aktualizujemy system_prompt o mowe dane


    if 'dokumentów' in message:
        system = system_prompt + f'nOdpowiadaj wyłącznie na podstawie dostarczonych dokumentów. Jeśli nie masz informacji — powiedz, że nie wiesz'
    else:
         system = system_prompt

    # budujemy message dla naszego modelu wrzucamy system czyli nowy system prompt message i hostiry
    messages = [{'role':'system','content':system}] + history + [{'role':'user','content':message}]

    # budujemy model 
    response = myOpenAI.chat.completions.create(

        model='gpt-4.1-mini',
        messages=messages

    )
    # zwracamy odpowedz do reply ktory jest uzzywany przez evaluation oraz rerun

    reply = response.choices[0].message.content

    # teraz uruhcamiamy funkcje evaluator i przeakzujemy ją do modelu evalaution zeby ustrukuryzowała odpowiedz 2 medlu spradzajacaego czyli evalautor

    evaluation = evaluator(reply,message,history)

    # teraz tworzymy if'a który działa dzieki funkcji evalautor który przekazuje weryfiakcje odpowiedzi chat (1 model) do Evalaution ( który zwróci nam odpoowiedz w ułozony sposób)

    if evaluation.is_acceptable:
        print('odpowiedź jest akceptowalna')

    else:
        print('odpowiedź błędna')
        print(evaluation.feedback)
        #uruhcamiamy 3 funkcje rerun która raz jeszcze odpowiada na pytanie usera, bogatdza o feedbkack z funkcju evaluator i ulozna dane z niej dzięki evalaution
        #a jej odpowiwedz zwracamy do reply, tego samego co zawsze
        reply = rerun(reply,message,history,evaluation.feedback)
        
    return reply



In [54]:
history = [] # tworzymy history

In [55]:
# teraz mozemy rozmawiac zawsze ten sam schemat 1 budujemy wiadomosc

msg1 = 'hej, co robi pm w projekcie, opowiedz mi coś i zajrzyj do dokuemntów'

# wywołanie funkcji chat

reply1 = chat(msg1,history)

# dodanie naszego message i reply do history

history.append({'role':'user','content':msg1})
history.append({'role':'assistant','content':reply1})


# print 

print(reply1)

odpowiedź jest akceptowalna
Cześć! PM, czyli Kierownik Projektu (Project Manager), pełni kluczową rolę w zarządzaniu projektem. Na podstawie dostarczonych materiałów, oto co robi PM w projekcie:

### Role i zadania Kierownika Projektu:
1. **Planowanie projektu**  
   - Określa szczegółowe cele, zakres, harmonogram oraz budżet projektu.  
   - Tworzy podział pracy (WBS - Work Breakdown Structure), ustala kolejność zadań i przypisuje odpowiedzialności.  
   - Przygotowuje harmonogram (np. w formie wykresu Gantta), który porządkuje pracę zespołu i pomaga kontrolować postęp.

2. **Organizacja i realizacja**  
   - Organizuje i koordynuje prace zespołu projektowego.  
   - Dba o właściwą komunikację w projekcie – formalną i nieformalną (np. e-maile, spotkania, komunikatory).  
   - Monitoruje przebieg projektu, kontroluje czas, koszty, zakres i jakość realizowanych działań.

3. **Zarządzanie ryzykiem**  
   - Identyfikuje potencjalne ryzyka, ocenia ich prawdopodobieństwo i wpływ na projekt.